# 00 — Smoke Test : Environnement GPU & Modèle Pré-entraîné

Ce notebook vérifie que ton environnement cloud (**Google Colab** ou **Kaggle**) est prêt pour le projet `endo-depth` :
1. Vérifier la disponibilité du **GPU (CUDA)** et les versions des bibliothèques.
2. Exécuter un calcul PyTorch simple sur le GPU (ou CPU en local).
3. Charger un modèle pré-entraîné d'estimation de profondeur monoculaire (`depth-anything/Depth-Anything-V2-Small-hf`).
4. Lancer une inférence test sur une image synthétique et afficher le résultat.

### Avant de lancer sur le cloud
- **Google Colab** : `Exécution` -> `Modifier le type d'exécution` -> `GPU T4`
- **Kaggle** : panneau de droite `Settings` -> `Accelerator` -> `GPU T4 x2` (ou `P100`), et activer `Internet: On`.

In [ ]:
# Installe uniquement les paquets manquants (utile sur Colab/Kaggle)
import importlib.util

required = ["transformers", "timm", "yaml", "huggingface_hub"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print("Installation des paquets manquants :", missing)
    %pip install -q "transformers>=4.45.0" timm pyyaml huggingface_hub
else:
    print("Tous les paquets requis sont déjà installés.")

In [ ]:
import platform
import sys

import cv2
import numpy as np
import timm
import torch
import torchvision
import transformers

print(f"OS           : {platform.system()} {platform.release()}")
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"timm         : {timm.__version__}")
print(f"OpenCV       : {cv2.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"CUDA dispo ? : {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU détecté  : {gpu_name} ({vram_gb:.2f} Go VRAM)")
else:
    print("[Info] Aucun GPU détecté -> exécution sur CPU (normal en local, active CUDA sur Colab/Kaggle).")

In [ ]:
# Test rapide d'allocation et de calcul sur le périphérique détecté
a = torch.randn(512, 512, device=device)
b = torch.randn(512, 512, device=device)
c = a @ b
print(f"Calcul PyTorch réussi sur {device} | shape={tuple(c.shape)} | mean={c.mean().item():.4f}")

## Chargement d'un modèle pré-entraîné : Depth Anything V2 Small

- **Modèle** : `depth-anything/Depth-Anything-V2-Small-hf` (~25 millions de paramètres, donc léger pour un GPU gratuit T4).
- **Architecture** : un encodeur **Vision Transformer (DINOv2 ViT-S)** qui découpe l'image en petits patchs, suivi d'un décodeur **DPT** qui reconstruit une carte dense pixel par pixel.
- **Sortie** : le modèle prédit une **disparité / profondeur relative inverse** (plus la valeur est grande, plus la surface est proche de la caméra).

In [ ]:
import time
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
print(f"Chargement de {MODEL_ID}...")
t0 = time.time()

image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Modèle chargé en {time.time() - t0:.2f} s | Paramètres totaux : {total_params:,}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Image synthétique évoquant une cavité endoscopique (lumen sombre, bosse/polype, reflet spéculaire)
h, w = 384, 384
yy, xx = np.meshgrid(np.linspace(-1, 1, h), np.linspace(-1, 1, w), indexing="ij")
radius = np.sqrt(xx ** 2 + yy ** 2)

wall = np.clip(0.25 + 0.65 * (radius ** 0.8), 0.0, 1.0)
polyp = 0.25 * np.exp(-((xx - 0.32) ** 2 + (yy + 0.15) ** 2) / 0.015)
specular = 0.45 * np.exp(-((xx + 0.25) ** 2 + (yy - 0.25) ** 2) / 0.004)

r = np.clip(wall * 0.95 + polyp * 0.60 + specular, 0.0, 1.0)
g = np.clip(wall * 0.48 + polyp * 0.35 + specular, 0.0, 1.0)
b = np.clip(wall * 0.42 + polyp * 0.30 + specular, 0.0, 1.0)
rgb_uint8 = (np.stack([r, g, b], axis=-1) * 255).astype(np.uint8)
sample_image = Image.fromarray(rgb_uint8)

inputs = image_processor(images=sample_image, return_tensors="pt").to(device)
if torch.cuda.is_available():
    torch.cuda.synchronize()
t0 = time.time()

with torch.no_grad():
    outputs = model(**inputs)
    predicted_depth = outputs.predicted_depth

depth_resized = (
    torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=(h, w),
        mode="bicubic",
        align_corners=False,
    )
    .squeeze()
    .cpu()
    .numpy()
)

if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed_ms = (time.time() - t0) * 1000
print(f"Inférence terminée en {elapsed_ms:.1f} ms | min={depth_resized.min():.3f}, max={depth_resized.max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample_image)
axes[0].set_title("Image test synthétique (lumen + polype)")
axes[0].axis("off")

im = axes[1].imshow(depth_resized, cmap="inferno")
axes[1].set_title("Disparité relative (clair = proche)")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Sauvegarde persistante (Colab / Kaggle)

Les sessions gratuites Colab/Kaggle s'arrêtent après quelques heures. Pour ne pas perdre tes checkpoints :
- **Colab** : décommente la cellule ci-dessous pour monter Google Drive.
- **Kaggle** : sauvegarde tes fichiers dans `/kaggle/working/` et utilise `Save Version`.

In [ ]:
# Décommenter uniquement sur Google Colab :
# from google.colab import drive
# drive.mount('/content/drive')